# Модель кредитного  скоринга
### Кушнир Анна, МФТИ Прикладная математика и информатика


## Постановка задачи

Нужно редсказать, уйдёт ли клиент в дефолт по кредиту. У каждого клиента несколько кредитных продуктов, и нужно было из этой «панельной» структуры собрать один вектор признаков на человека.

Весь проект я делала в Google Colab с его ограничениями по памяти и времени. Пришлось искать баланс между качеством и тем, что вообще можно запустить


## Решение
Все решение находится в этом ноутбуке

* **Агрегация данных**. Я использовала two‑step подход: one‑hot кодирование всех признаков (кроме `id` и `rn`) с суммированием по клиенту + добавила среднее, максимум и минимум по всем числовым колонкам. Это позволило сохранить и категориальную информацию, и числовые статистики.

* **Модели**. Основной упор сделала на XGBoost и CatBoost. Параметры я настраивала через Optuna. LightGBM на моих данных показал слабый результат (0.719), поэтому я от него отказалась.

* **Ансамбль**. CatBoost показал лучший индивидуальный результат (0.7558). Я нашла оптимальные веса для блендинга с XGBoost — получилось 0.95 / 0.05. Это дало небольшое улучшение до 0.7563 на валидации.

* **Признаки**. Помимо базовой агрегации, я добавила ранговые признаки: логарифм `id`, ранг `id`, ранг количества продуктов и отношение `rn_max` к числу продуктов. Они должны были помочь модели уловить временные тренды.




## Ограничения

* Colab даёт около 12 ГБ RAM, и это жёсткое ограничение. Я не могла загрузить все данные целиком, поэтому читала их по частям (батчами по 250 000 строк). Это работало, но требовало аккуратности в агрегации. Optuna я запускала всего на 60 итераций, больше просто не выполнилось. CatBoost я обучала без Optuna, подбирала параметры руками.


## Итоговый результат

На валидации я получила **0.7563 AUC**. На тесте платформа показала **0.7314**  Расхождение говорит о том, что распределение теста отличается от валидации, и, возможно, стоило сделать временной разрез строже.



In [1]:
!pip install catboost optuna optuna-integration

In [2]:
from gc import collect
from scipy import stats
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold , cross_validate
from lightgbm import plot_importance
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from optuna.integration import CatBoostPruningCallback
from google.colab import drive
from pathlib import Path
import os
import shutil
import math
import pyarrow.parquet as pq

/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `optuna.integration.catboost` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.catboost` instead.
  return _bootstrap._gcd_import(name[level:], package, level)


## Подключение Google Drive и выбор объема данных




In [3]:
drive.mount('/content/drive')

DRIVE_TRAIN_DATA_PATH = Path('/content/drive/MyDrive/Копия train_data.parquet')
DRIVE_TRAIN_TARGET_PATH = Path('/content/drive/MyDrive/Копия train_target.csv')
LOCAL_WORK_DIR = Path('/content/credit_scoring_sample')

# Сколько строк признаков забрать из исходного parquet.
N_ROWS_TO_LOAD = 600_000

TEST_SHARE = 0.2

RECREATE_LOCAL_FILES = True


if RECREATE_LOCAL_FILES and LOCAL_WORK_DIR.exists():
    shutil.rmtree(LOCAL_WORK_DIR)

LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

assert DRIVE_TRAIN_DATA_PATH.exists(), f'Не найден файл: {DRIVE_TRAIN_DATA_PATH}'
assert DRIVE_TRAIN_TARGET_PATH.exists(), f'Не найден файл: {DRIVE_TRAIN_TARGET_PATH}'


def read_first_n_rows_from_parquet(parquet_path: Path, n_rows=None) -> pd.DataFrame:
    """
    Читает первые n_rows из parquet без загрузки всего файла целиком.
    Это нужно только на стартовом этапе, чтобы подготовить меньшие локальные файлы
    для старого ноутбука
    """
    parquet_file = pq.ParquetFile(parquet_path)

    if n_rows is None:
        return pd.read_parquet(parquet_path)

    batches = []
    rows_read = 0

    # batch_size можно уменьшить, если стартовая ячейка тоже упирается в RAM.
    for batch in parquet_file.iter_batches(batch_size=100_000):
        part = batch.to_pandas()
        need = n_rows - rows_read

        if len(part) > need:
            part = part.iloc[:need].copy()

        batches.append(part)
        rows_read += len(part)

        if rows_read >= n_rows:
            break

    if not batches:
        return pd.DataFrame()

    return pd.concat(batches, ignore_index=True)


sample_data = read_first_n_rows_from_parquet(
    DRIVE_TRAIN_DATA_PATH,
    n_rows=N_ROWS_TO_LOAD,
)

print('Loaded feature rows:', sample_data.shape)

# Читаем target и оставляем только id, которые реально попали в sample_data
target_all = pd.read_csv(DRIVE_TRAIN_TARGET_PATH)

sample_ids = pd.Series(sample_data['id'].unique(), name='id')
target_sample = target_all[target_all['id'].isin(sample_ids)].copy()

print('Loaded target rows after id filter:', target_sample.shape)

#  Делаем простой train/test split на уровне id, чтобы старые test_data/test_target тоже существовали
#  Если target сильно несбалансирован, стратификацию дальше все равно делает исходный train_test_split.
rng = pd.Series(target_sample['id'].unique()).sample(frac=1.0, random_state=42).reset_index(drop=True)

n_test_ids = int(len(rng) * TEST_SHARE)
test_ids = set(rng.iloc[:n_test_ids])
train_ids = set(rng.iloc[n_test_ids:])

train_data_sample = sample_data[sample_data['id'].isin(train_ids)].copy()
test_data_sample = sample_data[sample_data['id'].isin(test_ids)].copy()

target_train = target_sample[target_sample['id'].isin(train_ids)].copy()
target_test = target_sample[target_sample['id'].isin(test_ids)].copy()

print('train_data_sample:', train_data_sample.shape)
print('test_data_sample:', test_data_sample.shape)
print('target_train:', target_train.shape)
print('target_test:', target_test.shape)

train_chunks = []
n_train_chunks = 12

for i in range(n_train_chunks):
    start = math.floor(len(train_data_sample) * i / n_train_chunks)
    end = math.floor(len(train_data_sample) * (i + 1) / n_train_chunks)

    chunk = train_data_sample.iloc[start:end].copy()
    chunk_path = LOCAL_WORK_DIR / f'train_data_{i}.pq'
    chunk.to_parquet(chunk_path, index=False)

    train_chunks.append(chunk_path)

print('Saved train chunks:', len(train_chunks))

# Сохраняем test_data_0.pq и test_data_1.pq
n_test_chunks = 2

for i in range(n_test_chunks):
    start = math.floor(len(test_data_sample) * i / n_test_chunks)
    end = math.floor(len(test_data_sample) * (i + 1) / n_test_chunks)

    chunk = test_data_sample.iloc[start:end].copy()
    chunk_path = LOCAL_WORK_DIR / f'test_data_{i}.pq'
    chunk.to_parquet(chunk_path, index=False)

print('Saved test chunks:', n_test_chunks)

# Сохраняем target-файлы
target_train.to_csv(LOCAL_WORK_DIR / 'train_target.csv', index=False)
target_test.to_csv(LOCAL_WORK_DIR / 'test_target.csv', index=False)
os.chdir(LOCAL_WORK_DIR)

print('Current working directory:', Path.cwd())
print('Files prepared in:', LOCAL_WORK_DIR)

del sample_data, target_all, target_sample, train_data_sample, test_data_sample

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded feature rows: (600000, 61)
Loaded target rows after id filter: (65832, 2)
train_data_sample: (480472, 61)
test_data_sample: (119528, 61)
target_train: (52666, 2)
target_test: (13166, 2)
Saved train chunks: 12
Saved test chunks: 2
Current working directory: /content/credit_scoring_sample
Files prepared in: /content/credit_scoring_sample


# Импорт данных


In [4]:
train_data_0=pd.concat([pd.read_parquet('train_data_7.pq'),
                     pd.read_parquet('train_data_8.pq'),pd.read_parquet('train_data_9.pq'),
                     pd.read_parquet('train_data_10.pq'),pd.read_parquet('train_data_11.pq'),
                     ],ignore_index=True)

In [5]:
# training
train_data_1=pd.concat([pd.read_parquet('train_data_1.pq'),
                     pd.read_parquet('train_data_0.pq'),pd.read_parquet('train_data_2.pq'),
                     pd.read_parquet('train_data_3.pq'),pd.read_parquet('train_data_4.pq'),
                     pd.read_parquet('train_data_5.pq'),pd.read_parquet('train_data_6.pq'),
                     ],ignore_index=True)

target_train=pd.read_csv('train_target.csv')
target_test=pd.read_csv('test_target.csv')
test_id=pd.read_csv('test_target.csv')
test_data=pd.concat([pd.read_parquet('test_data_0.pq'),pd.read_parquet('test_data_1.pq')],ignore_index=True)
pd.set_option("display.max_rows", None, "display.max_columns", None)

In [6]:
# Исторические признаки (на основе rn)
def add_history_features(df):
    """Создаёт агрегированные статистики по кредитной истории (по rn)"""
    # Сортируем для корректного порядка
    df_sorted = df.sort_values(['id', 'rn'])

    # Группируем по id и считаем статистики
    agg_dict = {
        'rn': ['count', 'max', 'min'],
        'pre_loans_credit_limit': ['mean', 'max', 'min', 'std', 'last'],
        'pre_util': ['mean', 'max', 'min', 'std', 'last'],
        'pre_loans_outstanding': ['mean', 'max', 'min', 'std', 'last'],
        'pre_loans_total_overdue': ['sum', 'max', 'mean', 'last'],
        'pre_loans5': ['sum', 'max', 'last'],
        'pre_loans530': ['sum', 'max', 'last'],
        'pre_loans3060': ['sum', 'max', 'last'],
        'pre_loans6090': ['sum', 'max', 'last'],
        'pre_loans90': ['sum', 'max', 'last'],
        'pre_since_opened': ['mean', 'max', 'min', 'last'],
        'pre_till_pclose': ['mean', 'max', 'min', 'last'],
    }
    # Оставляем только те колонки, которые есть в df
    agg_dict = {k: v for k, v in agg_dict.items() if k in df.columns}

    history = df_sorted.groupby('id').agg(agg_dict)
    # Разворачиваем мультииндекс в плоские имена
    history.columns = ['_'.join(col).strip() for col in history.columns.values]
    return history.reset_index()

# Применяем к объединённым данным
train_data_full = pd.concat([train_data_0, train_data_1], ignore_index=True)
history_features = add_history_features(train_data_full)
print("Исторические признаки созданы:", history_features.shape)

# Сохраняем, чтобы потом объединить с агрегированными данными
# (пока просто держим в памяти)

Исторические признаки созданы: (52666, 46)


# Разведывательный анализ исходных данных

In [6]:
train_data_0.describe()
#train_data_1.describe()

,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,pre_loans_outstanding,pre_loans_total_overdue,pre_loans_max_overdue_sum,pre_loans_credit_cost_rate,pre_loans5,pre_loans530,pre_loans3060,pre_loans6090,pre_loans90,is_zero_loans5,is_zero_loans530,is_zero_loans3060,is_zero_loans6090,is_zero_loans90,pre_util,pre_over2limit,pre_maxover2limit,is_zero_util,is_zero_over2limit,is_zero_maxover2limit,enc_paym_0,enc_paym_1,enc_paym_2,enc_paym_3,enc_paym_4,enc_paym_5,enc_paym_6,enc_paym_7,enc_paym_8,enc_paym_9,enc_paym_10,enc_paym_11,enc_paym_12,enc_paym_13,enc_paym_14,enc_paym_15,enc_paym_16,enc_paym_17,enc_paym_18,enc_paym_19,enc_paym_20,enc_paym_21,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag
count,2.001970e+05,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.0,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.00000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000,200197.000000
mean,2.324368e+06,7.250918,9.469218,8.100041,8.365875,8.321563,7.178949,8.369831,9.657397,2.337742,2.989256,0.0,2.005505,4.746155,5.972992,15.878120,5.001429,3.999760,8.003731,0.923321,0.835652,0.956892,0.974395,0.975359,13.863549,2.235303,15.679870,0.700005,0.921932,0.860547,0.150212,0.366039,0.511716,0.635484,0.759402,0.887850,1.033712,1.193994,1.302422,1.409382,1.528759,2.668751,1.796835,1.933520,1.99979,2.051899,2.098768,2.142884,2.187655,2.230838,3.267217,2.299910,2.333202,2.368157,3.498939,1.036454,2.695305,3.608416,1.001424,0.136426,0.230933
std,1.129103e+04,5.513276,5.835960,4.745993,5.351536,4.488335,5.110000,4.232955,5.840455,1.255491,0.700527,0.0,0.256837,3.501798,0.408291,1.149494,0.081862,0.026819,0.159518,0.266083,0.370592,0.203100,0.157953,0.155028,4.314629,0.873336,3.973238,0.458256,0.268279,0.346419,0.605033,0.918578,1.079341,1.182785,1.267011,1.336155,1.396680,1.442063,1.463316,1.475826,1.480132,1.473094,1.454601,1.420216,1.39900,1.380104,1.361277,1.341289,1.319668,1.297011,1.276299,1.256604,1.235290,1.211882,1.109346,0.313820,0.494040,1.048000,0.043716,0.343241,0.421430
min,2.304754e+06,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.0,1.000000,0.000000,0.000000,0.000000,2.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.314677e+06,3.000000,4.000000,4.000000,4.000000,6.000000,2.000000,5.000000,4.000000,2.000000,3.000000,0.0,2.000000,2.000000,6.000000,16.000000,5.000000,4.000000,8.000000,1.000000,1.000000,1.000000,1.000000,1.000000,15.000000,2.000000,17.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,1.000000,3.000000,3.000000,3.000000,3.000000,4.000000,1.000000,2.000000,3.000000,1.000000,0.000000,0.000000
50%,2.324456e+06,6.000000,10.000000,9.

In [17]:
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# Объединяем тренировочные данные (у вас уже есть train_data_0 и train_data_1)
train_data_full = pd.concat([train_data_0, train_data_1], ignore_index=True)
print("Размер объединённых данных:", train_data_full.shape)

# Считаем количество кредитных продуктов (записей) для каждого клиента
count_ops = train_data_full.groupby('id')['rn'].count().reset_index(name='count')

# Отдельно для дефолтных и недефолтных клиентов
ids_no_def = target_train[target_train['flag'] == 0]['id'].values
ids_def = target_train[target_train['flag'] == 1]['id'].values

count_no_def = count_ops[count_ops['id'].isin(ids_no_def)]['count']
count_def = count_ops[count_ops['id'].isin(ids_def)]['count']

# Общий максимум для оси X (чтобы графики не съезжали)
max_val = max(count_no_def.max(), count_def.max()) + 2

print("Клиентов без дефолта:", len(count_no_def))
print("Клиентов с дефолтом:", len(count_def))

Размер объединённых данных: (480472, 61)
Клиентов без дефолта: 50988
Клиентов с дефолтом: 1678


In [18]:
train_data_0.info()
#train_data_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200197 entries, 0 to 200196
Data columns (total 61 columns):
 #   Column                         Non-Null Count   Dtype
---  ------                         --------------   -----
 0   id                             200197 non-null  int64
 1   rn                             200197 non-null  int64
 2   pre_since_opened               200197 non-null  int64
 3   pre_since_confirmed            200197 non-null  int64
 4   pre_pterm                      200197 non-null  int64
 5   pre_fterm                      200197 non-null  int64
 6   pre_till_pclose                200197 non-null  int64
 7   pre_till_fclose                200197 non-null  int64
 8   pre_loans_credit_limit         200197 non-null  int64
 9   pre_loans_next_pay_summ        200197 non-null  int64
 10  pre_loans_outstanding          200197 non-null  int64
 11  pre_loans_total_overdue        200197 non-null  int64
 12  pre_loans_max_overdue_sum      200197 non-null  int64
 13 

In [19]:
train_data_0.isna().sum()

,0
id,0
rn,0
pre_since_opened,0
pre_since_confirmed,0
pre_pterm,0
pre_fterm,0
pre_till_pclose,0
pre_till_fclose,0
pre_loans_credit_limit,0
pre_loans_next_pay_summ,0


In [20]:
color_box = '#1f77b4'      # синий
color_hist = '#2ca02c'     # зелёный
color_kde = '#d62728'      # красный

fig_no_def = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    x_title='<b>Количество кредитных продуктов</b>'
)

fig_no_def.update_layout(
    title={
        'text': '<b>Распределение количества продуктов<br>у клиентов БЕЗ дефолта</b>',
        'font': {'size': 30, 'family': 'Arial, sans-serif', 'color': '#2c3e50'},
        'x': 0.5
    },
    height=750,
    width=1000,
    bargap=0.15,
    template='plotly_white',
    font=dict(family='Arial, sans-serif', size=14)
)

# Boxplot
fig_no_def.add_box(
    x=count_no_def,
    y0=0,
    row=1, col=1,
    showlegend=False,
    boxmean=True,
    fillcolor=color_box,
    line=dict(color='black', width=1.5),
    name='',
    hoverinfo='x'
)
fig_no_def.update_yaxes(
    title={'text': 'Коробчатая диаграмма', 'font': {'size': 18}},
    ticklabelposition='inside top',
    row=1, col=1,
    showticklabels=False
)

# Гистограмма + KDE
dist_fig = ff.create_distplot(
    hist_data=[count_no_def],
    group_labels=[''],
    bin_size=1,
    show_rug=False,
    show_hist=True,
    histnorm='probability',
    colors=[color_hist]
)
for trace in dist_fig.select_traces():
    if trace.type == 'scatter':
        trace.line.color = color_kde
        trace.line.width = 3
    elif trace.type == 'bar':
        trace.marker.color = color_hist
        trace.opacity = 0.6

for trace in dist_fig.select_traces():
    fig_no_def.add_trace(trace, row=2, col=1)

fig_no_def.update_yaxes(
    title={'text': 'Плотность вероятности', 'font': {'size': 18}},
    row=2, col=1
)

# Вертикальные линии для среднего и медианы
mean_val = count_no_def.mean()
median_val = count_no_def.median()
fig_no_def.add_vline(x=mean_val, line_dash='dash', line_color='red', row=2, col=1,
                     annotation_text=f'Среднее = {mean_val:.1f}', annotation_position='top')
fig_no_def.add_vline(x=median_val, line_dash='dot', line_color='blue', row=2, col=1,
                     annotation_text=f'Медиана = {median_val:.1f}', annotation_position='bottom')

fig_no_def.update_xaxes(range=[0, max_val])
fig_no_def.show()

In [21]:
color_box_def = '#d62728'    # красный
color_hist_def = '#ff7f0e'   # оранжевый
color_kde_def = '#9467bd'    # фиолетовый

fig_def = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    x_title='<b>Количество кредитных продуктов</b>'
)

fig_def.update_layout(
    title={
        'text': '<b>Распределение количества продуктов<br>у клиентов С дефолтом</b>',
        'font': {'size': 30, 'family': 'Arial, sans-serif', 'color': '#2c3e50'},
        'x': 0.5
    },
    height=750,
    width=1000,
    bargap=0.15,
    template='plotly_white',
    font=dict(family='Arial, sans-serif', size=14)
)

fig_def.add_box(
    x=count_def,
    y0=0,
    row=1, col=1,
    showlegend=False,
    boxmean=True,
    fillcolor=color_box_def,
    line=dict(color='black', width=1.5),
    name='',
    hoverinfo='x'
)
fig_def.update_yaxes(
    title={'text': 'Коробчатая диаграмма', 'font': {'size': 18}},
    ticklabelposition='inside top',
    row=1, col=1,
    showticklabels=False
)

dist_fig_def = ff.create_distplot(
    hist_data=[count_def],
    group_labels=[''],
    bin_size=1,
    show_rug=False,
    show_hist=True,
    histnorm='probability',
    colors=[color_hist_def]
)
for trace in dist_fig_def.select_traces():
    if trace.type == 'scatter':
        trace.line.color = color_kde_def
        trace.line.width = 3
    elif trace.type == 'bar':
        trace.marker.color = color_hist_def
        trace.opacity = 0.6

for trace in dist_fig_def.select_traces():
    fig_def.add_trace(trace, row=2, col=1)

fig_def.update_yaxes(
    title={'text': 'Плотность вероятности', 'font': {'size': 18}},
    row=2, col=1
)

mean_def = count_def.mean()
median_def = count_def.median()
fig_def.add_vline(x=mean_def, line_dash='dash', line_color='red', row=2, col=1,
                  annotation_text=f'Среднее = {mean_def:.1f}', annotation_position='top')
fig_def.add_vline(x=median_def, line_dash='dot', line_color='blue', row=2, col=1,
                  annotation_text=f'Медиана = {median_def:.1f}', annotation_position='bottom')

fig_def.update_xaxes(range=[0, max_val])
fig_def.show()

In [22]:
fig_compare = go.Figure()

fig_compare.add_trace(go.Box(
    x=count_no_def,
    name='Без дефолта',
    boxmean=True,
    marker_color='#1f77b4',
    line=dict(color='black', width=1.5),
    fillcolor='#1f77b4',
    opacity=0.7
))

fig_compare.add_trace(go.Box(
    x=count_def,
    name='С дефолтом',
    boxmean=True,
    marker_color='#d62728',
    line=dict(color='black', width=1.5),
    fillcolor='#d62728',
    opacity=0.7
))

fig_compare.update_layout(
    title={
        'text': '<b>Сравнение количества кредитных продуктов<br>у клиентов с дефолтом и без</b>',
        'font': {'size': 28, 'family': 'Arial, sans-serif', 'color': '#2c3e50'},
        'x': 0.5
    },
    xaxis_title='Количество продуктов',
    height=600,
    width=900,
    template='plotly_white',
    font=dict(family='Arial, sans-serif', size=16),
    boxmode='group'
)

fig_compare.update_xaxes(range=[0, max_val])
fig_compare.show()

In [23]:
flag_counts = target_train['flag'].value_counts()
fig_pie = go.Figure(data=[go.Pie(
    labels=['Не дефолт', 'Дефолт'],
    values=flag_counts,
    hole=0.4,
    marker=dict(colors=['#2ca02c', '#d62728']),
    textinfo='label+percent',
    textfont=dict(size=16, family='Arial, sans-serif')
)])

fig_pie.update_layout(
    title={
        'text': '<b>Дисбаланс целевой переменной</b>',
        'font': {'size': 28, 'family': 'Arial, sans-serif', 'color': '#2c3e50'},
        'x': 0.5
    },
    height=500,
    width=600,
    template='plotly_white',
    annotations=[dict(text=f'Всего: {len(target_train):,} клиентов', x=0.5, y=0.5, font_size=18, showarrow=False)]
)
fig_pie.show()

### Аггрегирование категориальных признаков в сумму фиктивных переменных

In [7]:
def aggregations(data_frame: pd.DataFrame):
    # ---- 1. One-hot + сумма (как раньше) ----
    feature_cols = [col for col in data_frame.columns if col not in ['id', 'rn']]
    dummies = pd.get_dummies(data_frame[feature_cols], columns=feature_cols)
    dummy_features = dummies.columns.values
    ohe_features = pd.concat([data_frame, dummies], axis=1)
    ohe_features = ohe_features.drop(columns=feature_cols)
    sum_features = ohe_features.groupby('id')[dummy_features].sum().reset_index()

    # ---- 2. Новые производные признаки на уровне отдельных продуктов ----
    # Создаём копию, чтобы не портить исходный data_frame
    df = data_frame.copy()
    # Отношения (защита от деления на ноль)
    df['limit_to_outstanding'] = df['pre_loans_credit_limit'] / (df['pre_loans_outstanding'] + 1e-6)
    df['util_to_limit'] = df['pre_util'] / (df['pre_loans_credit_limit'] + 1e-6)
    df['overdue_to_limit'] = df['pre_loans_total_overdue'] / (df['pre_loans_credit_limit'] + 1e-6)
    df['overdue_to_outstanding'] = df['pre_loans_total_overdue'] / (df['pre_loans_outstanding'] + 1e-6)
    # Суммарная просрочка
    overdue_cols = ['pre_loans5', 'pre_loans530', 'pre_loans3060', 'pre_loans6090', 'pre_loans90']
    df['total_overdue'] = df[overdue_cols].sum(axis=1)
    df['any_overdue'] = (df['total_overdue'] > 0).astype(int)
    # Количество просроченных категорий
    df['n_overdue_types'] = (df[overdue_cols] > 0).sum(axis=1)

    # ---- 3. Статистики (mean, max, min) по ВСЕМ колонкам (включая новые) ----
    numeric_cols = [col for col in df.columns if col not in ['id', 'rn']]
    stats = df.groupby('id')[numeric_cols].agg(['mean', 'max', 'min']).reset_index()
    stats.columns = ['id'] + [f"{col}_{stat}" for col, stat in stats.columns[1:]]

    # ---- 4. Количество продуктов и статистики по rn ----
    rn_stats = df.groupby('id')['rn'].agg(['count', 'max', 'min']).reset_index()
    rn_stats.columns = ['id', 'product_count', 'rn_max', 'rn_min']

    # ---- 5. Объединяем всё ----
    combined = sum_features.merge(stats, on='id', how='left')
    combined = combined.merge(rn_stats, on='id', how='left')
    combined.fillna(0, inplace=True)
    return combined

In [8]:
features_0=aggregations(train_data_0)
train_df_0=target_train.merge(features_0,on='id')
feature_cols0=list(train_df_0.columns.values)
feature_cols0.remove("id"), feature_cols0.remove("flag")
y0=train_df_0['flag'].values

In [9]:
features_1=aggregations(train_data_1)
train_df_1=target_train.merge(features_1,on='id')
feature_cols1=list(train_df_1.columns.values)
feature_cols1.remove("id"), feature_cols1.remove("flag")
y1=train_df_1['flag'].values

In [10]:

train_df=pd.concat([train_df_0,train_df_1])
y0df=pd.DataFrame(y0,columns=['flag'])
y1df=pd.DataFrame(y1,columns=['flag'])
y=pd.concat([y0df,y1df])['flag'].values

In [11]:

train_df.fillna(0,inplace=True)

#Целевое кодирование

In [12]:
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np

def target_encoding(train_df, target_col, cat_cols, n_folds=5, alpha=100):
    """
    Целевое кодирование с гладкой оценкой и кросс-валидацией.
    alpha – параметр сглаживания (чем больше, тем сильнее тянет к среднему).
    """
    encoded = pd.DataFrame(index=train_df.index)
    y = train_df[target_col]
    prior = y.mean()

    for col in cat_cols:
        if col not in train_df.columns:
            print(f"Колонка {col} не найдена, пропускаем")
            continue
        kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
        encoded_col = pd.Series(index=train_df.index, dtype=float)

        for train_idx, val_idx in kf.split(train_df):
            agg = train_df.iloc[train_idx].groupby(col)[target_col].agg(['sum', 'count'])
            agg['encoded'] = (agg['sum'] + alpha * prior) / (agg['count'] + alpha)
            encoded_val = train_df.iloc[val_idx][col].map(agg['encoded']).fillna(prior)
            encoded_col.iloc[val_idx] = encoded_val

        encoded_col.fillna(prior, inplace=True)
        encoded[col + '_te'] = encoded_col

    return encoded

# Список категориальных колонок, которые есть в train_df
cat_cols = ['enc_loans_account_holder_type', 'enc_loans_credit_status',
            'enc_loans_credit_type', 'enc_loans_account_cur']

# Добавляем целевое кодирование
te_features = target_encoding(train_df, 'flag', cat_cols, n_folds=5, alpha=100)
train_df = pd.concat([train_df, te_features], axis=1)
print("Добавлены признаки целевого кодирования:", te_features.columns.tolist())
print("Размер train_df после добавления:", train_df.shape)

Колонка enc_loans_account_holder_type не найдена, пропускаем
Колонка enc_loans_credit_status не найдена, пропускаем
Колонка enc_loans_credit_type не найдена, пропускаем
Колонка enc_loans_account_cur не найдена, пропускаем
Добавлены признаки целевого кодирования: []
Размер train_df после добавления: (52667, 601)


### Сопоставляем данные колонки для свопадения размерности в train и test

In [14]:
features_sub=aggregations(test_data)


feature_cols_sub=list(features_sub.columns.values)
feature_cols_sub.remove('id')

feature_cols_both=list(set(feature_cols0) & set(feature_cols1) & set(feature_cols_sub))

X=train_df[feature_cols_both]

X_train, X_val, y_train, y_val = train_test_split(X, y,
                                                  test_size=0.2,stratify=y,
                                                  random_state=42)
del features_0, features_1,  train_df_0, train_df_1, train_data_0, train_data_1
collect()

30

In [15]:
# Добавляем те же ранговые признаки к тестовым данным
features_sub['log_id'] = np.log1p(features_sub['id'])
features_sub['id_rank'] = features_sub['id'].rank(pct=True)
features_sub['product_count_rank'] = features_sub['product_count'].rank(pct=True)
features_sub['rn_span_ratio'] = features_sub['rn_max'] / (features_sub['product_count'] + 1)

print("Ранговые признаки добавлены в тест:", features_sub[['log_id', 'id_rank', 'product_count_rank', 'rn_span_ratio']].columns.tolist())

Ранговые признаки добавлены в тест: ['log_id', 'id_rank', 'product_count_rank', 'rn_span_ratio']


# XGboost

Перебор параметров


In [24]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# Если в X_train есть колонки с недопустимыми символами — очищаем
def clean_columns(df):
    df.columns = [str(col).replace('[', '_').replace(']', '_').replace('<', '_') for col in df.columns]
    return df

X_train_clean = clean_columns(X_train.copy())
X_val_clean = clean_columns(X_val.copy())

def objective(trial):
    # диапазоны для ключевых параметров
    params = {
        'device': 'cuda',
        'verbosity': 0,
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'eta': trial.suggest_float('eta', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
        'alpha': trial.suggest_float('alpha', 0.0, 5.0),
        'lambda': trial.suggest_float('lambda', 0.0, 5.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.5, 30, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 500, 3000),
        'seed': 42,
        'early_stopping_rounds': 30
    }

    model = XGBClassifier(**params)
    model.fit(
        X_train_clean, y_train,
        eval_set=[(X_val_clean, y_val)],
        verbose=False
    )
    preds = model.predict_proba(X_val_clean)[:, 1]
    auc = roc_auc_score(y_val, preds)
    return auc

# Создаём исследование и запускаем
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=10)

print("Лучшие параметры:", study.best_params)
print("Лучший AUC на валидации:", study.best_value)

[I 2026-06-19 16:12:05,703] A new study created in memory with name: no-name-0d01165a-51fe-426c-bc43-baca16f4d3b0
[I 2026-06-19 16:12:53,748] Trial 0 finished with value: 0.7314786279289123 and parameters: {'eta': 0.03574712922600244, 'max_depth': 12, 'subsample': 0.8659969709057025, 'colsample_bytree': 0.7993292420985183, 'min_child_weight': 2, 'gamma': 0.15599452033620265, 'alpha': 0.2904180608409973, 'lambda': 4.330880728874676, 'scale_pos_weight': 5.8592686909850995, 'n_estimators': 2270}. Best is trial 0 with value: 0.7314786279289123.
[I 2026-06-19 16:13:18,925] Trial 1 finished with value: 0.7388944435883787 and parameters: {'eta': 0.010725209743171997, 'max_depth': 12, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'min_child_weight': 2, 'gamma': 0.18340450985343382, 'alpha': 1.5212112147976886, 'lambda': 2.6237821581611893, 'scale_pos_weight': 2.9311198685320003, 'n_estimators': 1228}. Best is trial 1 with value: 0.7388944435883787.
[I 2026-06-19 16:1

Лучшие параметры: {'eta': 0.015019490572374374, 'max_depth': 10, 'subsample': 0.8803925243084487, 'colsample_bytree': 0.7806385987847482, 'min_child_weight': 8, 'gamma': 0.49379559636439074, 'alpha': 2.6136641469099704, 'lambda': 2.137705091792748, 'scale_pos_weight': 0.5548416522539616, 'n_estimators': 769}
Лучший AUC на валидации: 0.7523522352655516


In [16]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# Очистка имён колонок от недопустимых символов
def clean_columns(df):
    df.columns = [str(col).replace('[', '_').replace(']', '_').replace('<', '_') for col in df.columns]
    return df

X_train_clean = clean_columns(X_train.copy())
X_val_clean = clean_columns(X_val.copy())

# Лучшие параметры от Optuna
best_params = {
    'eta': 0.01812895267923747,
    'max_depth': 3,
    'subsample': 0.9409477633811673,
    'colsample_bytree': 0.5177662169674748,
    'min_child_weight': 10,
    'gamma': 0.18770963308151886,
    'alpha': 3.678810955569957,
    'lambda': 4.147620905588646,
    'scale_pos_weight': 5.806462226508028,
    'n_estimators': 2677,
    'device': 'cuda',
    'verbosity': 0,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'seed': 42,
    'early_stopping_rounds': 50
}

# Создаём и обучаем модель
model2 = XGBClassifier(**best_params)
model2.fit(
    X_train_clean, y_train,
    eval_set=[(X_val_clean, y_val)],
    verbose=True
)

# Оценка качества
auc_val = roc_auc_score(y_val, model2.predict_proba(X_val_clean)[:, 1])
print(f"AUC на валидации: {auc_val:.6f}")

[0]	validation_0-auc:0.64860
[1]	validation_0-auc:0.67321
[2]	validation_0-auc:0.69047
[3]	validation_0-auc:0.68621
[4]	validation_0-auc:0.69495
[5]	validation_0-auc:0.70060
[6]	validation_0-auc:0.70025
[7]	validation_0-auc:0.70488
[8]	validation_0-auc:0.70359
[9]	validation_0-auc:0.70487
[10]	validation_0-auc:0.70909
[11]	validation_0-auc:0.71002
[12]	validation_0-auc:0.70901
[13]	validation_0-auc:0.71084
[14]	validation_0-auc:0.71237
[15]	validation_0-auc:0.71514
[16]	validation_0-auc:0.71566
[17]	validation_0-auc:0.71534
[18]	validation_0-auc:0.71599
[19]	validation_0-auc:0.71518
[20]	validation_0-auc:0.71411
[21]	validation_0-auc:0.71632
[22]	validation_0-auc:0.71754
[23]	validation_0-auc:0.71777
[24]	validation_0-auc:0.71949
[25]	validation_0-auc:0.72122
[26]	validation_0-auc:0.72199
[27]	validation_0-auc:0.72380
[28]	validation_0-auc:0.72367
[29]	validation_0-auc:0.72295
[30]	validation_0-auc:0.72338
[31]	validation_0-auc:0.72345
[32]	validation_0-auc:0.72412
[33]	validation_0-au

# LightGBM

In [17]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

# Очистка имён (если ещё не сделана)
def clean_columns(df):
    df.columns = [str(col).replace('[', '_').replace(']', '_').replace('<', '_') for col in df.columns]
    return df

X_train_clean = clean_columns(X_train.copy())
X_val_clean = clean_columns(X_val.copy())

# Параметры LightGBM (теперь без device='gpu', используем CPU)
lgb_params = {
    'n_estimators': 1500,
    'learning_rate': 0.03,
    'num_leaves': 31,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 1.0,
    'reg_lambda': 2.0,
    'min_child_samples': 20,
    'scale_pos_weight': 5.8,   # возьмите ваше значение или 1
    'verbosity': -1,
    'random_state': 42,
    'n_jobs': -1   # использовать все ядра CPU
}

model_lgb = lgb.LGBMClassifier(**lgb_params)
model_lgb.fit(
    X_train_clean, y_train,
    eval_set=[(X_val_clean, y_val)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
)

# Предсказания LightGBM
pred_lgb = model_lgb.predict_proba(X_val_clean)[:, 1]
auc_lgb = roc_auc_score(y_val, pred_lgb)
print(f"LightGBM AUC (val): {auc_lgb:.6f}")

# Предсказания XGBoost (ваша уже обученная модель)
pred_xgb = model2.predict_proba(X_val_clean)[:, 1]
auc_xgb = roc_auc_score(y_val, pred_xgb)
print(f"XGBoost AUC (val): {auc_xgb:.6f}")

# Простой блендинг
pred_ensemble = (pred_xgb + pred_lgb) / 2
auc_ensemble = roc_auc_score(y_val, pred_ensemble)
print(f"Ensemble AUC (val): {auc_ensemble:.6f}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[3]	valid_0's binary_logloss: 0.137993
LightGBM AUC (val): 0.719441
XGBoost AUC (val): 0.752885
Ensemble AUC (val): 0.753994


# catboost + XGB


In [18]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

# Параметры CatBoost
cat_params = {
    'iterations': 800,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3,
    'verbose': True,
    'early_stopping_rounds': 50,
    'random_seed': 42,
    'thread_count': -1
}

cat_model = CatBoostClassifier(**cat_params)
cat_model.fit(
    X_train_clean, y_train,
    eval_set=(X_val_clean, y_val),
    use_best_model=True
)

# Предсказания CatBoost
pred_cat = cat_model.predict_proba(X_val_clean)[:, 1]
auc_cat = roc_auc_score(y_val, pred_cat)
print(f"CatBoost AUC (val): {auc_cat:.6f}")

# Предсказания XGBoost (уже есть)
pred_xgb = model2.predict_proba(X_val_clean)[:, 1]
auc_xgb = roc_auc_score(y_val, pred_xgb)
print(f"XGBoost AUC (val): {auc_xgb:.6f}")

# Ансамбль XGB + CatBoost
pred_ensemble = (pred_xgb + pred_cat) / 2
auc_ensemble = roc_auc_score(y_val, pred_ensemble)
print(f"Ensemble XGB+Cat AUC (val): {auc_ensemble:.6f}")

0:	learn: 0.6161290	test: 0.6162143	best: 0.6162143 (0)	total: 148ms	remaining: 1m 58s
1:	learn: 0.5494451	test: 0.5494826	best: 0.5494826 (1)	total: 226ms	remaining: 1m 30s
2:	learn: 0.4928807	test: 0.4929174	best: 0.4929174 (2)	total: 295ms	remaining: 1m 18s
3:	learn: 0.4443343	test: 0.4443115	best: 0.4443115 (3)	total: 365ms	remaining: 1m 12s
4:	learn: 0.4025255	test: 0.4025468	best: 0.4025468 (4)	total: 432ms	remaining: 1m 8s
5:	learn: 0.3663679	test: 0.3663427	best: 0.3663427 (5)	total: 525ms	remaining: 1m 9s
6:	learn: 0.3353232	test: 0.3353322	best: 0.3353322 (6)	total: 607ms	remaining: 1m 8s
7:	learn: 0.3089406	test: 0.3089480	best: 0.3089480 (7)	total: 686ms	remaining: 1m 7s
8:	learn: 0.2861514	test: 0.2862037	best: 0.2862037 (8)	total: 780ms	remaining: 1m 8s
9:	learn: 0.2670867	test: 0.2671524	best: 0.2671524 (9)	total: 844ms	remaining: 1m 6s
10:	learn: 0.2501744	test: 0.2502405	best: 0.2502405 (10)	total: 916ms	remaining: 1m 5s
11:	learn: 0.2359095	test: 0.2360559	best: 0.236

In [27]:
import numpy as np
from sklearn.metrics import roc_auc_score

best_auc = 0
best_weight_cat = 0.5
best_weight_xgb = 0.5

for w_cat in np.arange(0.5, 1.01, 0.05):
    w_xgb = 1 - w_cat
    pred_ens = w_cat * pred_cat + w_xgb * pred_xgb
    auc = roc_auc_score(y_val, pred_ens)
    if auc > best_auc:
        best_auc = auc
        best_weight_cat = w_cat
        best_weight_xgb = w_xgb

print(f"Лучший вес CatBoost: {best_weight_cat:.2f}")
print(f"Лучший вес XGBoost:   {best_weight_xgb:.2f}")
print(f"Лучший AUC ансамбля:  {best_auc:.6f}")


Лучший вес CatBoost: 0.95
Лучший вес XGBoost:   0.05
Лучший AUC ансамбля:  0.756294


In [ ]:
# Подготовка финального ансамбля для отправки

import time
import numpy as np
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score


# 2. Очистка имён колонок
def clean_columns(df):
    df.columns = [str(col).replace('[', '_').replace(']', '_').replace('<', '_') for col in df.columns]
    return df

X_clean = clean_columns(X.copy())

# Параметры моделей
cat_params_final = {
    'iterations': 1000,
    'learning_rate': 0.03,
    'depth': 7,
    'l2_leaf_reg': 10,
    'verbose': True,
    'random_seed': 42,
    'thread_count': -1,
    'eval_metric': 'AUC'
}

xgb_params_final = {
    'eta': 0.017719805419957706,
    'max_depth': 3,
    'subsample': 0.7841451834356706,
    'colsample_bytree': 0.5581569764539639,
    'min_child_weight': 1,
    'gamma': 0.9929032753198671,
    'alpha': 0.23305169421577604,
    'lambda': 3.462727239342195,
    'scale_pos_weight': 1.4617829906874893,
    'n_estimators': 3000,
    'device': 'cuda',
    'verbosity': 1,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'seed': 42
}

# Обучение CatBoost
start_time = time.time()
cat_final = CatBoostClassifier(**cat_params_final)
cat_final.fit(X_clean, y)


# Обучение XGBoost
start_time = time.time()
xgb_final = XGBClassifier(**xgb_params_final)
xgb_final.fit(X_clean, y)

# Ансамбль
ensemble_model = EnsembleModel(cat_final, xgb_final, weight1=0.95, weight2=0.05)
try:
    pred_ens_val = ensemble_model.predict_proba(X_val_clean)[:, 1]
    auc_ens_val = roc_auc_score(y_val, pred_ens_val)
    print(f"Ансамбль AUC на валидации: {auc_ens_val:.6f}")
except NameError:
    pass

# Переопределяем MODEL_FOR_SUBMISSION
MODEL_FOR_SUBMISSION = ensemble_model
def make_features_for_prediction(test_chunk: pd.DataFrame, feature_cols_both):
    features_sub_part = aggregations(test_chunk)

    # Ранговые признаки
    features_sub_part['log_id'] = np.log1p(features_sub_part['id'])
    features_sub_part['id_rank'] = features_sub_part['id'].rank(pct=True)
    features_sub_part['product_count_rank'] = features_sub_part['product_count'].rank(pct=True)
    features_sub_part['rn_span_ratio'] = features_sub_part['rn_max'] / (features_sub_part['product_count'] + 1)

    missing_cols = [col for col in feature_cols_both if col not in features_sub_part.columns]
    for col in missing_cols:
        features_sub_part[col] = 0

    X_sub_part = features_sub_part[feature_cols_both].fillna(0)
    return features_sub_part[['id']], X_sub_part

# Отправка результатов

Когда дошла до этапа предсказаний на тесте, я столкнулась с нехваткой памяти в Colab. Если попытаться загрузить тестовый файл `test_data.parquet` целиком, Colab просто упадет с ошибкой Out of Memory.

## Чтение по кусочкам

Я читаю тестовый файл по частям (батчам) по 250 000 строк. Это позволяет не загружать всё в память сразу. Но если просто резать файл на куски, можно разорвать одного клиента (его записи окажутся в разных батчах). А агрегация (группировка по id) требует, чтобы все записи одного клиента были вместе. Иначе у меня получатся неправильные признаки.

Поэтому в каждом батче я нахожу последний id и переношу все его записи в следующий батч. Так я гарантирую, что каждый клиент обрабатывается целиком, без разрывов.


## Повторяю ту же агрегацию

Для каждого батча я применяю ровно ту же функцию aggregations, что использовала при обучении:

* One-hot кодирование и суммирование по клиенту
* Добавление средних, максимумов, минимумов
* Добавление количества продуктов и статистик по rn

Потом я оставляю только те колонки, которые были в обучении (feature_cols_both). Если вдруг в батче нет какой-то колонки (редкая категория), я просто добавляю её со значением 0, иначе модель выдаст ошибку.


## Предсказываю и сохраняю по частям

Каждый батч я прогоняю через обученную модель XGBoost (model2), получаю вероятности дефолта и сохраняю результат в отдельный CSV-файл (submission_part_0000.csv, submission_part_0001.csv и т.д.). Так я не храню все предсказания в памяти — они сразу улетают на диск.


## В конце склеиваю всё вместе

Когда все батчи обработаны, я читаю все эти CSV-файлы и объединяю их в один большой DataFrame. Проверяю, нет ли дублирующихся id — если есть (а такое бывает, если алгоритм переноса последнего id сработал неидеально), я оставляю только последнее предсказание.


## Финальный файл

Всё это сохраняется в submission.csv. Файл содержит ровно столько строк, сколько уникальных клиентов в тесте



In [26]:
DRIVE_TEST_DATA_PATH = Path('/content/drive/MyDrive/Копия test_data.parquet')

SUBMISSION_DIR = Path('/content/submission_parts')
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

FINAL_SUBMISSION_PATH = Path('/content/submission.csv')


TEST_ROW_BATCH_SIZE = 250_000
RN_CUTOFF = None
PRED_COL = 'flag'
MODEL_FOR_SUBMISSION = model2


def predict_proba_positive(model, X):
    """
    Универсальная функция для получения вероятности класса 1
    """
    if hasattr(model, 'predict_proba'):
        pred = model.predict_proba(X)
        if len(pred.shape) == 2:
            return pred[:, 1]

        return pred

    # fallback, если модель возвращает сразу вероятности
    return model.predict(X)


def make_features_for_prediction(test_chunk: pd.DataFrame, feature_cols_both):
    features_sub_part = aggregations(test_chunk)

    # Добавляем отсутствующие OHE-колонки нулями
    # В каждом чанке могут встретиться не все категории
    missing_cols = [
        col for col in feature_cols_both
        if col not in features_sub_part.columns
    ]

    for col in missing_cols:
        features_sub_part[col] = 0

    X_sub_part = features_sub_part[feature_cols_both].fillna(0)

    return features_sub_part[['id']], X_sub_part


def iter_test_chunks_without_splitting_last_id(
    parquet_path: Path,
    batch_size: int,
    id_col: str = 'id',
):
    """
    Читает parquet кусками, но не разрывает последний id чанка
    """
    parquet_file = pq.ParquetFile(parquet_path)

    carry = pd.DataFrame()
    part_idx = 0

    for batch in parquet_file.iter_batches(batch_size=batch_size):
        chunk = batch.to_pandas()

        if not carry.empty:
            chunk = pd.concat([carry, chunk], ignore_index=True)
            carry = pd.DataFrame()

        if chunk.empty:
            continue

        # Последний id переносим в следующий чанк,
        # чтобы не агрегировать его частично
        last_id = chunk[id_col].iloc[-1]

        process_mask = chunk[id_col] != last_id
        process_chunk = chunk.loc[process_mask].copy()
        carry = chunk.loc[~process_mask].copy()

        if not process_chunk.empty:
            yield part_idx, process_chunk
            part_idx += 1

        del chunk, process_chunk
        gc.collect()

    # В самом конце обрабатываем накопленный последний id
    if not carry.empty:
        yield part_idx, carry


submission_part_paths = []

for part_idx, test_chunk in iter_test_chunks_without_splitting_last_id(
    parquet_path=DRIVE_TEST_DATA_PATH,
    batch_size=TEST_ROW_BATCH_SIZE,
    id_col='id',
):
    print(f'Processing test part {part_idx}: raw rows = {len(test_chunk):,}')

    # Применяем тот же rn cutoff, что и в train, если он был
    if RN_CUTOFF is not None:
        test_chunk = test_chunk[test_chunk['rn'] <= RN_CUTOFF].copy()

    if test_chunk.empty:
        continue

    ids_part, X_sub_part = make_features_for_prediction(
        test_chunk=test_chunk,
        feature_cols_both=feature_cols_both,
    )

    preds_part = predict_proba_positive(
        MODEL_FOR_SUBMISSION,
        X_sub_part,
    )

    submission_part = ids_part.copy()
    submission_part[PRED_COL] = preds_part

    part_path = SUBMISSION_DIR / f'submission_part_{part_idx:04d}.csv'
    submission_part.to_csv(part_path, index=False)

    submission_part_paths.append(part_path)
    del test_chunk, ids_part, X_sub_part, preds_part, submission_part
    gc.collect()

submission = pd.concat(
    [pd.read_csv(path) for path in submission_part_paths],
    ignore_index=True,
)

duplicate_count = submission['id'].duplicated().sum()

if duplicate_count > 0:
    submission = (
        submission
        .sort_values('id')
        .drop_duplicates(subset=['id'], keep='last')
        .reset_index(drop=True)
    )

submission.to_csv(FINAL_SUBMISSION_PATH, index=False)

print('Final submission saved to:', FINAL_SUBMISSION_PATH)
print('Final submission shape:', submission.shape)
display(submission.head())

Final submission saved to: /content/submission.csv
Final submission shape: (900000, 2)


In [25]:
from pathlib import Path
import pandas as pd

# Папка, куда сохранялись частичные submission-файлы
SUBMISSION_DIR = Path('/content/submission_parts')

# Итоговый файл для загрузки
FINAL_SUBMISSION_PATH = Path('/content/submission.csv')

# Находим все части submission
submission_part_paths = sorted(SUBMISSION_DIR.glob('submission_part_*.csv'))

print(f'Found submission parts: {len(submission_part_paths)}')

if len(submission_part_paths) == 0:
    raise FileNotFoundError(f'No submission_part_*.csv files found in {SUBMISSION_DIR}')

# Читаем и объединяем
submission = pd.concat(
    [pd.read_csv(path) for path in submission_part_paths],
    ignore_index=True
)

print('Rows before duplicate check:', len(submission))
print('Columns:', submission.columns.tolist())

# Проверяем дубли id
duplicate_count = submission['id'].duplicated().sum()
print('Duplicate ids:', duplicate_count)

# Если из-за чанков появились дубли id, оставляем последнюю версию
if duplicate_count > 0:
    submission = (
        submission
        .drop_duplicates(subset=['id'], keep='last')
        .reset_index(drop=True)
    )

print('Rows after duplicate check:', len(submission))

# Сохраняем финальный файл
submission.to_csv(FINAL_SUBMISSION_PATH, index=False)

print(f'Final submission saved to: {FINAL_SUBMISSION_PATH}')
display(submission.head())

Found submission parts: 33
Rows before duplicate check: 900000
Columns: ['id', 'flag']
Duplicate ids: 0
Rows after duplicate check: 900000
Final submission saved to: /content/submission.csv


,id,flag
0,2250005,0.139501
1,2250013,0.158198
2,2250014,0.058373
3,2250015,0.110162
4,2250020,0.108266


Found submission parts: 33
Rows before duplicate check: 900000
Columns: ['id', 'flag']
Duplicate ids: 0
Rows after duplicate check: 900000
Final submission saved to: /content/submission.csv


,id,flag
0,2250005,0.139501
1,2250013,0.158198
2,2250014,0.058373
3,2250015,0.110162
4,2250020,0.108266
